In [17]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
import statsmodels.formula.api as smf

In [18]:
def load_and_prepare_run(
    run_id,
    result_dir=""
):
    """
    Load loss, label, FOIF and TracIn results for one experimental run.

    Expected filenames:
        IF_sp1_de01_seed_<run_id>.csv
        TC_sp1_de01_seed_<run_id>.csv
        loss_sp1_de01_seed_<run_id>.csv
        label_sp1_de01_seed_<run_id>.csv
    """

    foif_df = pd.read_csv(
        f"IF_sp3_de001_run{run_id}.csv"
    )
    tracin_df = pd.read_csv(
        f"TC_sp3_de001_run{run_id}.csv"
    )
    loss_df = pd.read_csv(
        f"loss_sp3_de001_run{run_id}.csv"
    )
    label_df = pd.read_csv(
        f"label_sp3_de001_run{run_id}.csv"
    )

    label_df = label_df[
        ["id", "label", "cluster_id"]
    ].rename(columns={"id": "Train_ID"})

    foif_df = foif_df.rename(
        columns={"Score": "FOIF_Score"}
    )

    tracin_df = tracin_df.rename(
        columns={"Score": "TracIn_Score"}
    )

    density_df = (
        label_df
        .merge(loss_df, on="Train_ID", validate="one_to_one")
        .merge(foif_df, on="Train_ID", validate="one_to_one")
        .merge(tracin_df, on="Train_ID", validate="one_to_one")
    )

    density_df["Density_Group"] = density_df["cluster_id"].replace({
        "1_sparse": "Sparse",
        "0_sparse": "Sparse",
        "1_dense": "Dense",
        "0_dense": "Dense"
    })

    density_df["Sparse"] = (
        density_df["cluster_id"]
        .str.contains("sparse", case=False, na=False)
        .astype(int)
    )

    density_df["Log_Loss"] = np.log1p(
        density_df["Training_Loss"]
    )

    density_df["Log_Abs_TC"] = np.log1p(
        np.abs(density_df["TracIn_Score"])
    )

    density_df["Run"] = run_id

    return density_df

In [19]:
def fit_regression_for_run(density_df, run_id):
    """
    Fit signed-score and magnitude regression models for one run.
    """

    signed_model = smf.ols(
        "TracIn_Score ~ Log_Loss * Sparse",
        data=density_df
    ).fit()

    magnitude_model = smf.ols(
        "Log_Abs_TC ~ Log_Loss * Sparse",
        data=density_df
    ).fit()

    signed_result = {
        "Run": run_id,
        "Model": "Signed TC",
        "Beta_0_Intercept": signed_model.params["Intercept"],
        "Beta_1_Log_Loss": signed_model.params["Log_Loss"],
        "Beta_2_Sparse": signed_model.params["Sparse"],
        "Beta_3_Interaction": signed_model.params["Log_Loss:Sparse"],
        "Dense_Slope": signed_model.params["Log_Loss"],
        "Sparse_Slope": (
            signed_model.params["Log_Loss"]
            + signed_model.params["Log_Loss:Sparse"]
        ),
    }

    magnitude_result = {
        "Run": run_id,
        "Model": "Absolute TC magnitude",
        "Beta_0_Intercept": magnitude_model.params["Intercept"],
        "Beta_1_Log_Loss": magnitude_model.params["Log_Loss"],
        "Beta_2_Sparse": magnitude_model.params["Sparse"],
        "Beta_3_Interaction": magnitude_model.params[
            "Log_Loss:Sparse"
        ],
        "Dense_Slope": magnitude_model.params["Log_Loss"],
        "Sparse_Slope": (
            magnitude_model.params["Log_Loss"]
            + magnitude_model.params["Log_Loss:Sparse"]
        ),
    }

    return signed_result, magnitude_result

In [20]:
seeds = [1,2,3,4,5]

all_regression_results = []
all_run_data = []

for seed in seeds:
    print(f"Processing run {seed}...")

    density_df = load_and_prepare_run(
        run_id=seed,
        result_dir=""
    )

    signed_result, magnitude_result = fit_regression_for_run(
        density_df=density_df,
        run_id=seed
    )

    all_regression_results.extend([
        signed_result,
        magnitude_result
    ])

    all_run_data.append(density_df)

regression_results_df = pd.DataFrame(
    all_regression_results
)

all_density_df = pd.concat(
    all_run_data,
    ignore_index=True
)

display(regression_results_df)

Processing run 1...
Processing run 2...
Processing run 3...
Processing run 4...
Processing run 5...


,Run,Model,Beta_0_Intercept,Beta_1_Log_Loss,Beta_2_Sparse,Beta_3_Interaction,Dense_Slope,Sparse_Slope
0,1,Signed TC,0.123993,-2.774862,-0.110375,2.736746,-2.774862,-0.038116
1,1,Absolute TC magnitude,0.123404,-2.761426,-0.121662,2.786647,-2.761426,0.025221
2,2,Signed TC,-0.101570,2.266481,0.109450,-2.288904,2.266481,-0.022423
3,2,Absolute TC magnitude,-0.062304,1.467744,0.067628,-1.443721,1.467744,0.024023
4,3,Signed TC,-0.008690,0.267154,0.017067,-0.289894,0.267154,-0.022740
5,3,Absolute TC magnitude,-0.008654,0.266227,0.010862,-0.249353,0.266227,0.016874
6,4,Signed TC,-0.078983,1.870333,0.090780,-1.902433,1.870333,-0.032100
7,4,Absolute TC magnitude,-0.078601,1.861333,0.082126,-1.838842,1.861333,0.022491
8,5,Signed TC,-0.139162,3.185349,0.150303,-3.216281,3.185349,-0.030932
9,5,Absolute TC magnitude,-0.138587,3.172339,0.140890,-3.150259,3.172339,0.022080


In [21]:
coefficient_columns = [
    "Beta_0_Intercept",
    "Beta_1_Log_Loss",
    "Beta_2_Sparse",
    "Beta_3_Interaction",
    "Dense_Slope",
    "Sparse_Slope",
]

regression_summary = (
    regression_results_df
    .groupby("Model")[coefficient_columns]
    .agg(["mean", "std"])
)

display(regression_summary)

Beta_0_Intercept           Beta_1_Log_Loss            \
                                  mean       std            mean       std   
Model                                                                        
Absolute TC magnitude        -0.032948  0.098921        0.801243  2.245494   
Signed TC                    -0.040882  0.103699        0.962891  2.340778   

                      Beta_2_Sparse           Beta_3_Interaction            \
                               mean       std               mean       std   
Model                                                                        
Absolute TC magnitude      0.035969  0.099523          -0.779105  2.246239   
Signed TC                  0.051445  0.102533          -0.992153  2.337455   

                      Dense_Slope           Sparse_Slope            
                             mean       std         mean       std  
Model                                                               
Absolute TC magnitude    0.801243  2.245494     0.022138  0.003198  
Signed TC                0.962891  2.340778    -0.029262  0.006681

In [ ]:
# density_df["Loss_Decile"] = pd.qcut(
#     density_df["Training_Loss"],
#     q=10,
#     labels=False,
#     duplicates="drop"
# ) + 1

In [ ]:
# decile_label_summary = (
#     density_df
#     .groupby(["Loss_Decile", "Density_Group"])
#     .agg(
#         Count=("Train_ID", "size"),
#         Mean_Loss=("Training_Loss", "mean"),

#         Mean_FOIF=("FOIF_Score", "mean"),
#         FOIF_Positive_Fraction=(
#             "FOIF_Score",
#             lambda x: (x > 0).mean()
#         ),

#         Mean_TracIn=("TracIn_Score", "mean"),
#         TracIn_Positive_Fraction=(
#             "TracIn_Score",
#             lambda x: (x > 0).mean()
#         )
#     )
#     .reset_index()
# )

# print(decile_label_summary)